In [ ]:

"""
Dissertation Pipeline - STAGE: Data Acquisition & Harmonization
Author: Aamul sapkota
Description: This notebook handles the ETL (Extract, Transform, Load) process for the multi-source
landslide inventory. It ingests the NASA GLC dataset (Dataset A) and the Nepalese national databases
(Datasets B & C). It performs hierarchical deduplication and uses open-source shapefiles to
spatially geocode text-based ward records into explicit coordinates for the Random Forest model.
"""

# ENVIRONMENT SETUP & DEPENDENCIES
# Note: Run these shell commands first if executing in a fresh Colab instance.
# ===> for that, UNCOMMENT the following section thats before 'import'

"""
!pip install pandas geopandas shapely -q
!git clone https://github.com/SaugatPdl/nepal-administrative-boundary-shapefiles.git
!zip -r nepal-admin-boundaries.zip nepal-administrative-boundary-shapefiles
"""

import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import random
from google.colab import drive

# Mount Drive for exporting the final master dataset
drive.mount('/content/drive')

print(f"✅ Environment ready. GeoPandas version: {gpd.__version__}")



Cloning into 'nepal-administrative-boundary-shapefiles'...
remote: Enumerating objects: 57, done.
remote: Counting objects: 100% (57/57), done.
remote: Compressing objects: 100% (50/50), done.
remote: Total 57 (delta 7), reused 16 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (57/57), 19.48 MiB | 8.28 MiB/s, done.
Resolving deltas: 100% (7/7), done.
Encountered 8 file(s) that should have been pointers, but weren't:
	District/3_NepalDistricts.dbf
	District/3_NepalDistricts.shp
	Municipality/4_NepalMunicipality.dbf
	Municipality/4_NepalMunicipality.shp
	National_Boundary/1_NepalBoundary.dbf
	National_Boundary/1_NepalBoundary.shp
	Province/2_NepalProvince.dbf
	Province/2_NepalProvince.shp
  adding: nepal-administrative-boundary-shapefiles/ (stored 0%)
  adding: nepal-administrative-boundary-shapefiles/Province/ (stored 0%)
  adding: nepal-administrative-boundary-shapefiles/Province/2_NepalProvince.shp (deflated 24%)
  adding: nepal-administrative-boundary-shapefiles/Province/2

In [ ]:

# PHASE 1: STUDY AREA DEFINITION & SHAPEFILE GEOPROCESSING

# Defining the 13 specific districts of the Bagmati Province.
# Uppercase is used strictly to prevent string-matching errors later.
bagmati_districts = [
    'BHAKTAPUR', 'CHITAWAN', 'DHADING', 'DOLAKHA', 'KATHMANDU',
    'KAVREPALANCHOK', 'LALITPUR', 'MAKWANPUR', 'NUWAKOT',
    'RAMECHHAP', 'RASUWA', 'SINDHULI', 'SINDHUPALCHOK'
]

print("Loading datasets and standardizing spatial projections...")
# Load the official MoFAGA-derived Ward shapefile
wards_gdf = gpd.read_file('/content/nepal-administrative-boundary-shapefiles/Ward/5_NepalWards.shp')

# CRITICAL GEOPROCESSING STEP:
# Converting the shapefile to standard GPS Degrees (EPSG:4326 - WGS84).
# This ensures perfect compatibility when we export these coordinates to Google Earth Engine later.
wards_gdf = wards_gdf.to_crs("EPSG:4326")

# Clean shapefile text strings to ensure flawless table-joining
wards_gdf['DISTRICT'] = wards_gdf['DISTRICT'].astype(str).str.upper().str.strip()
wards_gdf['GaPa_NaPa'] = wards_gdf['GaPa_NaPa'].astype(str).str.upper().str.strip()

# Convert the shapefile ward numbers to integers (handles messy float cases like '5.0' -> 5)
wards_gdf['NEW_WARD_N'] = pd.to_numeric(wards_gdf['NEW_WARD_N'], errors='coerce').fillna(-1).astype(int)

# Isolate the Bagmati Province to create our regional bounding mask
bagmati_wards_gdf = wards_gdf[wards_gdf['DISTRICT'].isin(bagmati_districts)]
bagmati_boundary = bagmati_wards_gdf.unary_union # Merges all wards into a single master provincial polygon


# PHASE 2: TEMPORAL & SPATIAL FILTERING (Dataset A - NASA GLC)

# Load the raw CSVs from Google Drive
df_a = pd.read_csv('/content/drive/MyDrive/dissertation/landslide1_GLC.csv')
df_b = pd.read_csv('/content/drive/MyDrive/dissertation/landslide2_BIPAD.csv')
df_c = pd.read_csv('/content/drive/MyDrive/dissertation/landslide3_NDRRP.csv')

# Standardize dates across all three datasets to handle varying source formats
df_a['event_date'] = pd.to_datetime(df_a['event_date'], errors='coerce')
df_b['event_date'] = pd.to_datetime(df_b['event_date'], errors='coerce')
df_c['event_date'] = pd.to_datetime(df_c['event_date'], errors='coerce')

print("Processing Dataset A (GLC - High Precision)...")
# Isolate the 2010-2025 study period
df_a = df_a[(df_a['event_date'] >= '2010-01-01') & (df_a['event_date'] <= '2025-12-31')]

# Convert pandas dataframe to a GeoDataFrame using the exact latitude/longitude provided by NASA
gdf_a = gpd.GeoDataFrame(
    df_a,
    geometry=gpd.points_from_xy(df_a['longitude'], df_a['latitude']),
    crs="EPSG:4326"
)

# Spatial Filter: Keep only the coordinates that physically fall inside the Bagmati boundary
gdf_a = gdf_a[gdf_a.geometry.within(bagmati_boundary)]
gdf_a['source'] = 'Dataset_A_GLC'






Loading datasets and standardizing spatial projections...


/tmp/ipykernel_9100/4280011691.py:29: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  bagmati_boundary = bagmati_wards_gdf.unary_union # Merges all wards into a single master provincial polygon


Processing Dataset A (GLC - High Precision)...


/tmp/ipykernel_9100/4280011691.py:41: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_b['event_date'] = pd.to_datetime(df_b['event_date'], errors='coerce')
/tmp/ipykernel_9100/4280011691.py:42: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_c['event_date'] = pd.to_datetime(df_c['event_date'], errors='coerce')


In [ ]:

# PHASE 3: HARMONIZATION & HIERARCHICAL DEDUPLICATION (Datasets B & C)

print("Processing Datasets B & C (Administrative Text Records)...")

# Clean Dataset C: Filter to Bagmati districts and drop records lacking spatial precision (wards)
df_c['district'] = df_c['district'].astype(str).str.upper().str.strip()
df_c = df_c[df_c['district'].isin(bagmati_districts)]
df_c = df_c.dropna(subset=['municipality', 'ward'])

# Clean Dataset B: Ensure it has ward-level precision
df_b['district'] = df_b['district'].astype(str).str.upper().str.strip()
df_b = df_b.dropna(subset=['municipality', 'ward'])

# Merge the national portals and apply the temporal filter
df_bc = pd.concat([df_b, df_c], ignore_index=True)
df_bc = df_bc[(df_bc['event_date'] >= '2010-01-01') & (df_bc['event_date'] <= '2025-12-31')]

# Format municipality and ward strings for perfect matching
df_bc['municipality'] = df_bc['municipality'].astype(str).str.upper().str.strip()
df_bc['ward'] = pd.to_numeric(df_bc['ward'], errors='coerce').fillna(-1).astype(int)

# EXECUTE HIERARCHICAL DEDUPLICATION
# To prevent model overfitting on duplicate reports of the same event, we drop duplicates
# based on exact matches of Date, District, Municipality, and Ward.
initial_len = len(df_bc)
df_bc = df_bc.drop_duplicates(subset=['event_date', 'district', 'municipality', 'ward'], keep='first')
print(f"Deduplication complete: Removed {initial_len - len(df_bc)} duplicate events across Datasets B and C.")


Processing Datasets B & C (Administrative Text Records)...
Deduplication complete: Removed 171 duplicate events across Datasets B and C.


In [ ]:

# PHASE 4: SPATIAL GEOCODING VIA RANDOM POINT GENERATION

print("Generating spatial coordinates within Ward boundaries...")

def generate_random_point(polygon):
    """
    Generates a representative random coordinate strictly bounded within a ward polygon.
    Because ward boundaries in Nepal can be highly irregular, a while-loop is used to
    ensure the generated point is mathematically inside the polygon, not just its bounding box.
    """
    minx, miny, maxx, maxy = polygon.bounds
    while True:
        p = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))
        if polygon.contains(p):
            return p

valid_rows = []
generated_points = []
unmatched_count = 0

for index, row in df_bc.iterrows():
    # Identify the matching ward polygon in the official shapefile
    # Note: District + Ward Number provides a universally unique identifier in Nepal's structure
    match = bagmati_wards_gdf[
        (bagmati_wards_gdf['DISTRICT'] == row['district']) &
        (bagmati_wards_gdf['NEW_WARD_N'] == row['ward'])
    ]

    if not match.empty:
        ward_poly = match.geometry.values[0]
        rand_pt = generate_random_point(ward_poly)
        generated_points.append(rand_pt)
        valid_rows.append(row)
    else:
        unmatched_count += 1

print(f"Successfully geocoded coordinates for {len(valid_rows)} events.")
if unmatched_count > 0:
    print(f" Note: {unmatched_count} events were dropped due to unresolvable spelling mismatches with the official Shapefile.")

# Convert successfully geocoded administrative records into a spatial GeoDataFrame
df_bc_valid = pd.DataFrame(valid_rows)
gdf_bc = gpd.GeoDataFrame(df_bc_valid, geometry=generated_points, crs="EPSG:4326")
gdf_bc['source'] = 'Dataset_BC_Admin'
gdf_bc['latitude'] = gdf_bc.geometry.y
gdf_bc['longitude'] = gdf_bc.geometry.x


# PHASE 5: MASTER DATABASE ASSEMBLY

print("Merging datasets into the finalized master spatial inventory...")

# Combine the exact coordinates (Dataset A) with the geocoded coordinates (Datasets B/C).
# As established in the methodology, no cross-deduplication is performed between A and BC
# due to their disparate spatial reporting structures.
final_columns = ['event_date', 'latitude', 'longitude', 'source']
master_df = pd.concat([gdf_a[final_columns], gdf_bc[final_columns]], ignore_index=True)

# Sort chronologically for historical continuity
master_df = master_df.sort_values(by='event_date').reset_index(drop=True)

# Export the final dataset
output_filename = '01master_landslide_inventory_bagmati.csv'
master_df.to_csv(output_filename, index=False)


print(f"IPELINE COMPLETE! Master dataset saved locally within runtime with {len(master_df)} unique spatial landslide events.")

Generating spatial coordinates within Ward boundaries...
Successfully geocoded coordinates for 1367 events.
⚠️ Note: 337 events were dropped due to unresolvable spelling mismatches with the official Shapefile.
Merging datasets into the finalized master spatial inventory...
✅ PIPELINE COMPLETE! Master dataset saved with 1419 unique spatial landslide events.
